# Train the Hangman BiLSTM + Attention on Kaggle GPU

Clones the `approach/bilstm-attention` branch of the project repo and runs
training there -- the model segfaults on this machine's CPU-only PyTorch
build locally (a Windows OpenMP/MKL threading conflict), and training is
much faster on GPU anyway.

This branches off `approach/bilstm` (which validated at 47.7% win rate)
with one architectural change: a self-attention layer between the BiLSTM
output and the classification head, so every position gets a direct,
weighted view of every other position instead of only what survives the
recurrence. Same training/validation/submission scripts otherwise --
directly comparable to the plain BiLSTM run.

**Before running:** in the notebook's Settings panel (right sidebar), set
**Accelerator = GPU T4 x2** (or any GPU) and **Internet = On** (needed to `git clone`).

In [ ]:
import torch
print('torch', torch.__version__, 'cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU detected -- check Settings > Accelerator in the sidebar')

In [ ]:
REPO_URL = "https://github.com/Sahoo-Achyutananda/MELTWATER---HACKATHON.git"
BRANCH = "approach/bilstm-attention"

!rm -rf repo
!git clone --branch $BRANCH --single-branch $REPO_URL repo
%cd repo/brand-buzzword-hackathon
!ls

## Train

Same masked-language-model objective as the plain BiLSTM branch: randomly
mask letters in each training word, predict the true letter at each masked
position from bidirectional context. At inference: feed the real board
mask through the model, sum per-position letter probabilities across all
blanks, guess the highest-scoring unguessed letter.

In [ ]:
!python src/train_bilstm.py --epochs 20

## Validate

Same methodology as every other branch, for a fair comparison: hold out
10% of train.txt, play full interactive games against words the model
never trained on. Compare this number directly against the plain BiLSTM's
47.7% to see whether attention actually earns its keep here.

In [ ]:
!python src/validate_bilstm.py

## Generate submission.csv

Plays the actual game against every word in test.txt using the model
still in this session. Sandbox leaderboard checkpoint only -- per the
competition's Final Judgement policy, final hiring decisions re-run the
submitted model/notebook against a separate private word list.

250,000 words, one game at a time (not batched) -- prints progress every
20,000 words with an ETA. The plain BiLSTM branch's equivalent run took
~46 minutes end to end; expect something similar, maybe a bit more given
the extra attention layer per forward pass.

In [ ]:
!python src/generate_submission_bilstm.py

## Save outputs

Anything under `/kaggle/working/` is downloadable from the notebook's
Output tab after the run finishes.

In [ ]:
import shutil
shutil.copy("src/bilstm_attn_masker.pt", "/kaggle/working/bilstm_attn_masker.pt")
shutil.copy("submission.csv", "/kaggle/working/submission.csv")
print("saved bilstm_attn_masker.pt and submission.csv to /kaggle/working/ -- download from the Output tab")